# Gold Layer: Feature Engineering and Model Preparation

This notebook applies strict data leakage rules, removes direct churn signals, derives tenure/duration features from dates, and constructs a clean, isolated `model_ready_data.csv`.

In [28]:
from importlib import reload
import sys
from pathlib import Path
BASE_DIR = Path().resolve().parents[1]
sys.path.append(str(BASE_DIR))
print("Project Root:", BASE_DIR)

Project Root: C:\Users\sjram\OneDrive\Documents\customer-churn-analysis


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import classification_report, accuracy_score
from src.models.train import train_all_models, get_best_model

In [30]:
PROCESSED_DATA = BASE_DIR / "data" / "03_processed"

df = pd.read_parquet(PROCESSED_DATA / "final_dataset.parquet")

print(df.shape)
df.head()

(21526, 33)


,customer_account_number,lost_agreements,active_agreements,customer_churn,account_number,product_bob,fee_bob,total_revenue,service_interval,avg_unit_amount,...,bpg,msdyn_product_number,product_name,billing_period,machine,machine_variant,chemistry,total_agreements,revenue_per_agreement,service_intensity
0,IE01-C2042480-L,2,0,1,Unknown,0.0,0.0,0.0,0.0,0.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0
1,IE01-C2048834-L,0,2,0,Unknown,0.0,0.0,0.0,0.0,0.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0
2,IE01-C2156428-L,0,4,0,Unknown,0.0,0.0,0.0,0.0,0.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0
3,IE01-C2244318-L,0,2,0,Unknown,0.0,0.0,0.0,0.0,0.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0
4,IE01-C2270501-L,0,10,0,Unknown,0.0,0.0,0.0,0.0,0.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.0,0.0,0.0


In [31]:
df.columns

Index(['customer_account_number', 'lost_agreements', 'active_agreements',
       'customer_churn', 'account_number', 'product_bob', 'fee_bob',
       'total_revenue', 'service_interval', 'avg_unit_amount',
       'billing_interval', 'company_sizing', 'postal_code', 'branch',
       'vat_number', 'agreement_number', 'agreement_start_date',
       'agreement_end_date', 'renewal_type', 'agreement_type',
       'line_of_business', 'system_status', 'is_bob', 'bpg',
       'msdyn_product_number', 'product_name', 'billing_period', 'machine',
       'machine_variant', 'chemistry', 'total_agreements',
       'revenue_per_agreement', 'service_intensity'],
      dtype='object')

In [32]:
df = df.drop(columns=[
    'resolution_status',
    'lost_agreements',
    'active_agreements'
], errors='ignore')

In [33]:
y = df['customer_churn']   # 0,1,2
X = df.drop(columns=['customer_churn'])

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

print("Num cols:", len(num_cols))
print("Cat cols:", len(cat_cols))

Num cols: 9
Cat cols: 21


In [36]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

In [48]:
models = {
    "Logistic": (
        LogisticRegression(max_iter=1000),
        {"model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["lbfgs"],
}
    ),

    "RandomForest": (
        RandomForestClassifier(),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [5, 10, 20, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"]
        }
    ),

    "GradientBoosting": (
        GradientBoostingClassifier(),
        {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5],
            "model__subsample": [0.8, 1.0]
        }
    )
}

In [49]:
def create_pipeline(model):
    return Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

def run_model(model, params,preprocessor,X_train,y_train):

    pipe = create_pipeline(model)

    grid = GridSearchCV(
        pipe,
        params,
        cv=5,
        scoring="f1_macro",   # IMPORTANT for multi-class
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    return grid

def train_all_models(models_dict, preprocessor, X_train, y_train):

    results = {}

    for name, (model, params) in models_dict.items():
        print(f"\nTraining {name}...")

        grid = run_model(model, params, preprocessor, X_train, y_train)

        results[name] = grid

    return results

def get_best_model(results):

    best_model_name = max(results, key=lambda x: results[x].best_score_)
    best_model = results[best_model_name]

    return best_model_name, best_model

In [ ]:
results = train_all_models(models, preprocessor, X_train, y_train)


Training Logistic...

Training RandomForest...


In [ ]:
feature_names = best_model.best_estimator_.named_steps['preprocessing'].get_feature_names_out()

importances = best_model.best_estimator_.named_steps['model'].feature_importances_

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feat_imp.head(10)

In [ ]:
for name, model in results.items():
    print(f"\n{name}")
    print("Best Params:", model.best_params_)
    print("Score:", model.best_score_)

In [ ]:
best_model_name, best_model = get_best_model(results)

print("Best Model:", best_model_name)

In [ ]:
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))